- Title page
- Table of Contents
- Problem statement and its short explanation of the goal
- Methodology / problem-solving approach
- Key points of your technique/algorithms/ideas for speed-up
- Limitations and challenges encountered
- Test/example functions used, with graphical visualizations and output results
- Computational cost and time-complexity analysis (if you can)
- Ideas for further improvements of the projects
- References

## Tensor

## Convolutional Neural Network
[Convolutional neural network](https://en.wikipedia.org/wiki/Convolutional_neural_network) (CNN or ConvNet) is a well-established deep learning model. It specializes in processing spatially structured data like images. Its most notable feature is its use of convolutional kernels or filters. Kernels carry the trainable weights of the model, they go over the images and generates feature maps that capture some certain features of the original images, and they will be optimized during the training.

This design is actually inspired by the human visual system, each kernel in CNN serves as a visual neuron, while every neuron can only detect a small part of the visual field that is its [receptive field](https://en.wikipedia.org/wiki/Receptive_field). Furthermore, neurophysiologists [David H. Hubel](https://en.wikipedia.org/wiki/David_H._Hubel) and [Torsten Wiesel](https://en.wikipedia.org/wiki/Torsten_Wiesel) have revealed that human visual system is hierarchical, low-level neurons detect simple features like edges, lines or circles, while higher-level neurons integrate the information in order to recognize more complex patterns like cars, houses or people.

Compare to other deep learning model, CNN is known for its shift invariance, which means it doesn't matter where a certain pattern like a car appears in the image, it is able to recognize it. Another advantage of CNN is that it often requires fewer parameters due to its shared-weight architecture as the same kernel is reused for the whole image, unlike linear model that every pixel of the image is assigned with a unique parameter, which can lead to a very high memory-cost and problems like vanishing gradients or exploding gradients. For example, consider an input image of size 100 × 100 with 3 channels (RGB), each parameter of the next layer requires 100 × 100 × 3 = 30,000 weights in any fully connected neural network, while it only needs 10 × 10 × 3 = 300 weights in a CNN with a kernel size of 10 × 10.

In this chapter we will cover the following topics:
- Introduction of 2D convolution operation.
- Algorithms and some linear algebra used in the forward pass.
- Algorithms and some simple matrix calculus used in the backward pass.
- Example of training on [MNIST](https://en.wikipedia.org/wiki/MNIST_database) dataset.

### 2D Convolution Operation
In functional analysis, [convolution](https://en.wikipedia.org/wiki/Convolution) is a mathematical operation that takes two functions $f(x)$ and $g(x)$, and produces a new function $(f * g)(x)$ by first reflecting one of the functions over the y-axis, then sliding it over the other function, summing the products of their values at each point where they overlap. For example, reflecting $g(x)$ gives $g(-x)$. Then, sliding $g(-x)$ over $f(x)$ by a displacement of $t$ and multiplying their values gives $f(x)g(t - x)$. Finally, summing the products by taking the integral gives us its definition:
$$
(f * g)(t) := \int_{-\infty}^\infty f(\tau) g(t - \tau) \, d\tau.
$$
As convolution is commutative, we have an equivalent definition:
$$
(f * g)(t) := \int_{-\infty}^\infty f(t - \tau) g(\tau)\, d\tau.
$$

However, in the field of deep learning, the term convolution is often used to refer to [cross-correlation](https://en.wikipedia.org/wiki/Cross-correlation). For example, CNN actually uses cross-correlation instead of convolution. although the two operations are very similar, the only difference is that cross-correlation does not involve function reflection. For continuous functions $f$ and $g$, cross-correlation is defined as:
$$
(f \star g)(t) := \int_{-\infty}^\infty f(\tau) g(t + \tau) \, d\tau.
$$

For consistency, we will continue to use convolution to refer to the operation used in CNN. Furthermore, since images and kernels can be viewed as discrete functions in 2D, the discrete version of 2D convolution is defined as:
$$
(I * K)(i, j) := \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} I(i + m, j + n) K(m, n).
$$
- $(M, N)$: The shape of the kernel,
- $I(i, j)$: The value of pixel $(i, j)$ in the image,
- $K(m, n)$: The value of parameter $(m, n)$ in the kernel.

2D convolution is the core mathematical operation in CNN. The kernel acts as a sliding window over the image, summing the element-wise products between the kernel and the corresponding pixels of the image, like a sliding dot product. For example, given an image of shape $(5, 5)$ and a kernel of shape $(3, 3)$:
$$
I =
\begin{bmatrix}
0 & 1 & 1 & 1 & 0 \\
0 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 \\
0 & 0 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0
\end{bmatrix},
\quad
K =
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$

As you can see, the tiny binary image resembles a handwritten digit $9$, while the kernel is randomly initialized with natural numbers. According to the equation above, we have:
$$
\begin{align}
(I * K)(0, 0) &=
\begin{bmatrix}
0 & 1 & 1 \\
0 & 1 & 0 \\
0 & 1 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix} \\
&= \sum_{m=0}^{2} \sum_{n=0}^{2} I(m, n) K(m, n) \\
&= 27
\end{align}
$$

This is achieved by simply placing the kernel over the top-left corner of the image and taking the dot product. Similarly, we can compute the value of $(I * K)(1, 1)$ by sliding the kernel one step to the right and one step down, we have:
$$
\begin{align}
(I * K)(1, 1) &=
\begin{bmatrix}
1 & 0 & 1 \\
1 & 1 & 1 \\
0 & 0 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix} \\
&= \sum_{m=0}^{2} \sum_{n=0}^{2} I(1 + m, 1 + n) K(m, n) \\
&= 28
\end{align}
$$

Finally, we can obtain the entire output of shape $(3, 3)$ as follows:
$$
I * K =
\begin{bmatrix}
27 & 40 & 23 \\
13 & 28 & 19 \\
22 & 36 & 23
\end{bmatrix}
$$
